<img src="https://res.cloudinary.com/dtizipxds/image/upload/q_auto/f_auto/v1776264397/banner_yvwajv.png" width="100%">


In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn


# Solution - Anomaly Detection


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

sns.set_theme(style="whitegrid")
train_df = pd.read_csv(Path("../../Exercises/Anomaly Detection/train.csv"))
test_df = pd.read_csv(Path("../../Exercises/Anomaly Detection/test.csv"))
truth_df = pd.read_csv(Path("solution.csv"))
print(train_df.shape, test_df.shape, truth_df.shape)


In [ ]:
feature_cols = [c for c in train_df.columns if c.startswith("V")] + ["amount"]
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

z = np.abs((X_test - X_train.mean()) / X_train.std(ddof=0))
z_flag = (z > 3).any(axis=1).astype(int)

iso = IsolationForest(contamination=0.01, random_state=42)
iso.fit(X_train_scaled)
iso_flag = (iso.predict(X_test_scaled) == -1).astype(int)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
lof_flag = (lof.fit_predict(X_test_scaled) == -1).astype(int)


In [ ]:
res = truth_df.copy()
res["z_score_flag"] = z_flag
res["isolation_forest_flag"] = iso_flag
res["lof_flag"] = lof_flag

for col in ["z_score_flag", "isolation_forest_flag", "lof_flag"]:
    match = (res[col] == res["is_anomaly_true"]).mean()
    print(f"{col} accuracy vs truth: {match:.4f}")

res.head()


In [ ]:
plot_df = test_df[["time_s", "amount"]].copy()
plot_df["isolation_forest"] = iso_flag

plt.figure(figsize=(9,5))
sns.scatterplot(data=plot_df.sample(n=min(2000, len(plot_df)), random_state=42), x="time_s", y="amount", hue="isolation_forest", s=25, alpha=0.8)
plt.title("Detected anomalies (Isolation Forest)")
plt.tight_layout()
plt.show()
